# SO-101 VLA Data-Strategy Study: Training

Fine-tunes SmolVLA on each collected dataset and uploads the final checkpoint to the Hub.
Everything else in the study runs locally on a MacBook Air; only training runs here, on a
Colab A100.

Hyperparameters are fixed by PROTOCOL.md §4.7 and are identical for every run. The seed is
the only setting that varies between replications (§4.7, §8.14). The evaluated checkpoint is
the final one at step 10000 (§4.8), which the assert below enforces.

The training cell is a template: it was re-run once per row of the table below, changing
only `CONDITION`, `SEED` and `DATASET`.

In [ ]:
!nvidia-smi

In [ ]:
!pip install "lerobot[smolvla,dataset]"
!pip install av               # PyAV, needed to decode the recorded episode video

In [ ]:
# Records the training environment. Colab only; the versions here are what the runs used.
import lerobot, torch, transformers, platform, sys

print("\n".join([
    f"python       {sys.version.split()[0]}",
    f"platform     {platform.platform()}",
    f"lerobot      {lerobot.__version__}",
    f"torch        {torch.__version__}",
    f"transformers {transformers.__version__}",
    f"cuda         {torch.version.cuda}",
    f"gpu          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}",
]))

In [ ]:
from huggingface_hub import login
login()

import wandb
wandb.login()

HF_USER = "your-hf-username"

In [ ]:
from huggingface_hub import HfApi
for d in HfApi().list_datasets(author=HF_USER):
    print(d.id)

## Runs

| # | condition | seed | dataset | model repo |
|---|---|---|---|---|
| 1 | clean | 1000 | `cube-pickup-clean_20260809_105745` | `smolvla-cube-clean` |
| 2 | randomized | 1000 | `cube-pickup-randomized_20260809_115825` | `smolvla-cube-randomized` |
| 3 | recovery | 1000 | `cube-pickup-recovery_20260809_141725` | `smolvla-cube-recovery` |
| 4 | color | 1000 | `cube-pickup-color_20260809_183224` | `smolvla-cube-color` |
| 5 | clean | 2000 | `cube-pickup-clean_20260809_105745` | `smolvla-cube-clean-seed2000` |
| 6 | randomized | 2000 | `cube-pickup-randomized_20260809_115825` | `smolvla-cube-randomized-seed2000` |
| 7 | recovery | 2000 | `cube-pickup-recovery_20260809_141725` | `smolvla-cube-recovery-seed2000` |
| 8 | color | 2000 | `cube-pickup-color_20260809_183224` | `smolvla-cube-color-seed2000` |
| 9 | color-slowpace | 1000 | `cube-pickup-color_20260809_130649` | `smolvla-cube-color-slowpace` |

Runs 1 to 8 are the registered grid. Run 9 is the exploratory pace probe (PROTOCOL.md §8.16), trained on the superseded Color collection and never pooled into the grid. Set CONDITION = "color-slowpace" for it; the naming template then produces the right model repo and output paths on its own. Only the dataset name is irregular, since the slow-pace collection kept its original cube-pickup-color_ prefix.

In [ ]:
import os
import shutil
import subprocess
from huggingface_hub import HfApi

CONDITION = "randomized"
SEED      = 2000
DATASET   = f"{HF_USER}/cube-pickup-randomized_20260809_115825"
OUTDIR    = f"outputs/train/smolvla_{CONDITION}_seed{SEED}"
SUFFIX    = "" if SEED == 1000 else f"-seed{SEED}"
MODELREPO = f"{HF_USER}/smolvla-cube-{CONDITION}{SUFFIX}"

shutil.rmtree(OUTDIR, ignore_errors=True)

# Identical to the seed-1000 run except --seed and the output paths.
# Built as a Python string because IPython's {} expansion chokes on the
# JSON braces in --rename_map and silently passes the whole line through raw.
cmd = (
    "lerobot-train"
    " --policy.path=lerobot/smolvla_base"
    " --policy.push_to_hub=false"
    f" --dataset.repo_id={DATASET}"
    """ --rename_map='{"observation.images.overhead": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}'"""
    " --batch_size=32"
    " --steps=10000"
    " --save_freq=2000"
    f" --seed={SEED}"
    f" --output_dir={OUTDIR}"
    f" --job_name=smolvla_{CONDITION}_seed{SEED}"
    " --policy.device=cuda"
    " --wandb.enable=true"
)
print(cmd)
subprocess.run(cmd, shell=True, check=True)     # ~50 min

api = HfApi()
api.create_repo(MODELREPO, repo_type="model", exist_ok=True)

CKPT = f"{OUTDIR}/checkpoints/010000/pretrained_model"
assert os.path.isdir(CKPT), f"no step-10000 checkpoint at {CKPT} (training did not complete)"
api.upload_folder(folder_path=CKPT, repo_id=MODELREPO, repo_type="model")

print(f"uploaded {MODELREPO}")